# Import

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from joblib import Parallel, delayed

warnings.filterwarnings("ignore")

# Settings

In [ ]:
DATA_DIR = Path("../Binance_data/processed_parquet")

UNIVERSE_TICKER_CSV = "crypto_universe_output/final_tickers_formation_2022_top200.csv"

FILTER_OUTPUT_DIR = "crypto_universe_output"
Path(FILTER_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

START_DATE = "2023-01-01"
END_DATE = "2025-12-31"

N_CORES = 20


In [3]:
def end_of_day_if_midnight(end_date):
    ts = pd.Timestamp(end_date)
    if ts == ts.normalize():
        return ts + pd.Timedelta(hours=23, minutes=59)
    return ts


def load_tickers(csv_path):
    if not os.path.isfile(csv_path):
        raise FileNotFoundError(f"Ticker CSV not found: {csv_path}")

    df = pd.read_csv(csv_path)

    if "ticker" not in df.columns:
        raise ValueError("CSV must contain 'ticker' column")

    tickers = df["ticker"].astype(str).tolist()
    return tickers


def find_parquet_for_ticker(ticker, data_dir):
    candidates = [
        os.path.join(data_dir, f"{ticker}_1m.parquet"),
        os.path.join(data_dir, f"{ticker}.parquet"),
    ]

    for path in candidates:
        if os.path.isfile(path):
            return path

    raise FileNotFoundError(f"No parquet file found for ticker {ticker}")


def normalize_datetime_index(df):
    df = df.copy()

    if "datetime" in df.columns:
        df["datetime"] = pd.to_datetime(df["datetime"])
        df = df.set_index("datetime")

    elif "open_time" in df.columns:
        # open_time이 millisecond Unix timestamp라면 unit="ms"가 필요할 수 있음
        if np.issubdtype(df["open_time"].dtype, np.number):
            df["datetime"] = pd.to_datetime(df["open_time"], unit="ms")
        else:
            df["datetime"] = pd.to_datetime(df["open_time"])

        df = df.set_index("datetime")

    else:
        df.index = pd.to_datetime(df.index)

    return df.sort_index()


def load_one_ticker_close(ticker, data_dir, start, end):
    path = find_parquet_for_ticker(ticker, data_dir)

    df = pd.read_parquet(path)
    df = normalize_datetime_index(df)

    if "close" not in df.columns:
        raise ValueError(f"No close column for ticker {ticker}")

    s = df.loc[start:end, "close"].astype("float32")
    s.name = ticker

    return s


# Build 1-minute close price panel
def build_price_panel(tickers, data_dir, start_date, end_date, n_jobs=1):
    start = pd.Timestamp(start_date)
    end = end_of_day_if_midnight(end_date)

    if not os.path.isdir(data_dir):
        raise FileNotFoundError(f"DATA_DIR not found: {data_dir}")

    series_list = Parallel(n_jobs=n_jobs, verbose=5)(
        delayed(load_one_ticker_close)(ticker, data_dir, start, end)
        for ticker in tickers
    )

    price_panel = pd.concat(series_list, axis=1, sort=True)

    full_idx = pd.date_range(start=start, end=end, freq="1min")
    price_panel = price_panel.reindex(full_idx)
    price_panel.index.name = "datetime"

    return price_panel.astype("float32")

# Daily log-price panels and missingness diagnostic
def build_daily_log_price_panels(price_panel, tickers, start_date, end_date):
    start = pd.Timestamp(start_date)
    end = end_of_day_if_midnight(end_date)

    sliced = price_panel.loc[start:end, tickers]

    if (sliced <= 0).values.any():
        bad = sliced.columns[(sliced <= 0).any(axis=0)].tolist()
        raise ValueError(f"Non-positive prices found: {bad[:5]}")

    log_panel = np.log(sliced)

    daily_panels = {
        pd.Timestamp(day): group
        for day, group in log_panel.groupby(log_panel.index.normalize(), sort=True)
    }

    return daily_panels


def make_missingness_diagnostic(daily_panels):
    rows = []

    for day, frame in daily_panels.items():
        n_minutes = len(frame)

        obs_per_ticker = frame.notna().sum(axis=0)
        nan_per_ticker = n_minutes - obs_per_ticker

        row = {
            "date": day,
            "n_minutes_in_day": n_minutes,
            "min_obs_across_tickers": int(obs_per_ticker.min()),
            "max_obs_across_tickers": int(obs_per_ticker.max()),
            "n_tickers_with_zero_obs": int((obs_per_ticker == 0).sum()),
            # "missing_ratio_max": float(nan_per_ticker.max() / n_minutes),
            "total_nan_cells": int(nan_per_ticker.sum()),
        }

        rows.append(row)

    diag = pd.DataFrame(rows)
    diag = diag.sort_values("date").reset_index(drop=True)

    return diag


# Clean daily panels
def clean_daily_panels(
    daily_panels,
    manual_drop_dates=None,
):
    if manual_drop_dates is None:
        manual_drop_dates = []

    manual_drop_dates = set(pd.to_datetime(manual_drop_dates))

    cleaned = {}

    for day, frame in daily_panels.items():
        if day in manual_drop_dates:
            continue

        # 하루 내부에서만 forward fill
        # 하루 시작 부분 NaN은 ffill로 채워지지 않으므로 dropna로 제거
        cleaned_frame = frame.ffill(axis=0).dropna(axis=0, how="any")

        # 모든 종목이 동시에 존재하는 분만 남음
        if len(cleaned_frame) > 0:
            cleaned[day] = cleaned_frame

    return cleaned, manual_drop_dates

# Run Preprocessing

In [4]:
analysis_tickers = load_tickers(UNIVERSE_TICKER_CSV)
print(f"Loaded universe: {len(analysis_tickers)} tickers")

price_panel = build_price_panel(
    tickers=analysis_tickers,
    data_dir=DATA_DIR,
    start_date=START_DATE,
    end_date=END_DATE,
    n_jobs=N_CORES,
)

print(f"Price panel shape: {price_panel.shape}")
display(price_panel.head())

Loaded universe: 200 tickers


[Parallel(n_jobs=20)]: Using backend LokyBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done  32 tasks      | elapsed:    4.9s
[Parallel(n_jobs=20)]: Done 122 tasks      | elapsed:   12.7s
[Parallel(n_jobs=20)]: Done 200 out of 200 | elapsed:   18.1s finished


Price panel shape: (1578240, 200)


,BTCUSDT,ETHUSDT,GMTUSDT,BNBUSDT,LUNAUSDT,SOLUSDT,XRPUSDT,DOGEUSDT,APEUSDT,SHIBUSDT,...,VTHOUSDT,ONGUSDT,OXTUSDT,AMPUSDT,AVAUSDT,WANUSDT,ALCXUSDT,DFUSDT,OMUSDT,XNOUSDT
datetime,,,,,,,,,,,,,,,,,,,,,
2023-01-01 00:00:00,16543.669922,1196.130005,0.2303,246.300003,1.2610,9.98,0.3390,0.07032,3.637,0.000008,...,0.000904,0.2129,0.0675,0.00305,0.528,0.1800,13.7,0.0382,0.0282,0.637
2023-01-01 00:01:00,16539.310547,1196.089966,0.2303,246.300003,1.2629,10.01,0.3390,0.07033,3.635,0.000008,...,0.000905,0.2129,0.0675,0.00305,0.528,0.1801,13.7,0.0382,0.0282,0.637
2023-01-01 00:02:00,16536.429688,1195.849976,0.2299,246.199997,1.2609,10.01,0.3387,0.07024,3.635,0.000008,...,0.000905,0.2129,0.0675,0.00305,0.528,0.1805,13.7,0.0382,0.0282,0.637
2023-01-01 00:03:00,16533.650391,1195.819946,0.2300,246.000000,1.2609,10.01,0.3387,0.07018,3.634,0.000008,...,0.000905,0.2122,0.0675,0.00305,0.528,0.1805,13.7,0.0382,0.0282,0.637
2023-01-01 00:04:00,16535.380859,1196.319946,0.2306,246.199997,1.2602,10.00,0.3386,0.07020,3.632,0.000008,...,0.000905,0.2122,0.0674,0.00305,0.528,0.1805,13.7,0.0382,0.0282,0.635


In [5]:
daily_log_price_panels = build_daily_log_price_panels(
    price_panel=price_panel,
    tickers=analysis_tickers,
    start_date=START_DATE,
    end_date=END_DATE,
)

daily_log_price_diag = make_missingness_diagnostic(daily_log_price_panels)

print(f"Total calendar days: {len(daily_log_price_panels):,}")
display(daily_log_price_diag.head())

print("Top days by total_nan_cells:")
display(
    daily_log_price_diag
    .sort_values("total_nan_cells", ascending=False)
    .head(10)
)

Total calendar days: 1,096


,date,n_minutes_in_day,min_obs_across_tickers,max_obs_across_tickers,n_tickers_with_zero_obs,total_nan_cells
0,2023-01-01,1440,1440,1440,0,0
1,2023-01-02,1440,1440,1440,0,0
2,2023-01-03,1440,1440,1440,0,0
3,2023-01-04,1440,1440,1440,0,0
4,2023-01-05,1440,1440,1440,0,0


Top days by total_nan_cells:


,date,n_minutes_in_day,min_obs_across_tickers,max_obs_across_tickers,n_tickers_with_zero_obs,total_nan_cells
82,2023-03-24,1440,1360,1360,0,16000
45,2023-02-15,1440,1439,1440,0,14
44,2023-02-14,1440,1439,1440,0,7
1064,2025-11-30,1440,1440,1440,0,0
23,2023-01-24,1440,1440,1440,0,0
1080,2025-12-16,1440,1440,1440,0,0
1079,2025-12-15,1440,1440,1440,0,0
8,2023-01-09,1440,1440,1440,0,0
1095,2025-12-31,1440,1440,1440,0,0
0,2023-01-01,1440,1440,1440,0,0


In [6]:
# Binance 이슈 날짜 수동 제거
MANUAL_DROP_DATES = [
    "2023-03-24",
]

cleaned_daily_log_price_panels, dropped_dates = clean_daily_panels(
    daily_panels=daily_log_price_panels,
    manual_drop_dates=MANUAL_DROP_DATES,
)

print(f"Days before cleanup: {len(daily_log_price_panels)}")
print(f"Days after cleanup : {len(cleaned_daily_log_price_panels)}")
print(f"Dropped days       : {len(dropped_dates)}")

print("Dropped dates:")
print(sorted(pd.Timestamp(d).strftime("%Y-%m-%d") for d in dropped_dates)[:20])

Days before cleanup: 1096
Days after cleanup : 1095
Dropped days       : 1
Dropped dates:
['2023-03-24']


# Save

In [ ]:
# Save final 200-asset minute-level log-price panel

cleaned_log_price_panel = (
    pd.concat(cleaned_daily_log_price_panels.values(), axis=0)
    .sort_index()
    .astype("float32")
)

cleaned_log_price_panel.index.name = "datetime"

print("Final cleaned log-price panel shape:")
print(cleaned_log_price_panel.shape)

display(cleaned_log_price_panel.head())
display(cleaned_log_price_panel.tail())

save_path = os.path.join(
    FILTER_OUTPUT_DIR,
    "log_price_2023_2025_top200.parquet"
)

cleaned_log_price_panel.to_parquet(save_path)

print(f"Saved: {save_path}")

Final cleaned log-price panel shape:
(1576800, 200)


,BTCUSDT,ETHUSDT,GMTUSDT,BNBUSDT,LUNAUSDT,SOLUSDT,XRPUSDT,DOGEUSDT,APEUSDT,SHIBUSDT,...,VTHOUSDT,ONGUSDT,OXTUSDT,AMPUSDT,AVAUSDT,WANUSDT,ALCXUSDT,DFUSDT,OMUSDT,XNOUSDT
datetime,,,,,,,,,,,,,,,,,,,,,
2023-01-01 00:00:00,9.713758,7.086847,-1.468372,5.506550,0.231905,2.300583,-1.081755,-2.654699,1.291159,-11.724882,...,-7.008681,-1.546933,-2.695628,-5.792614,-0.638659,-1.714798,2.617396,-3.26492,-3.568433,-0.450986
2023-01-01 00:01:00,9.713495,7.086813,-1.468372,5.506550,0.233411,2.303585,-1.081755,-2.654557,1.290609,-11.724882,...,-7.007576,-1.546933,-2.695628,-5.792614,-0.638659,-1.714243,2.617396,-3.26492,-3.568433,-0.450986
2023-01-01 00:02:00,9.713321,7.086613,-1.470111,5.506144,0.231826,2.303585,-1.082641,-2.655837,1.290609,-11.724882,...,-7.007576,-1.546933,-2.695628,-5.792614,-0.638659,-1.712024,2.617396,-3.26492,-3.568433,-0.450986
2023-01-01 00:03:00,9.713153,7.086587,-1.469676,5.505332,0.231826,2.303585,-1.082641,-2.656692,1.290334,-11.724882,...,-7.007576,-1.550226,-2.695628,-5.792614,-0.638659,-1.712024,2.617396,-3.26492,-3.568433,-0.450986
2023-01-01 00:04:00,9.713258,7.087006,-1.467071,5.506144,0.231270,2.302585,-1.082936,-2.656407,1.289783,-11.723646,...,-7.007576,-1.550226,-2.697110,-5.792614,-0.638659,-1.712024,2.617396,-3.26492,-3.568433,-0.454130


,BTCUSDT,ETHUSDT,GMTUSDT,BNBUSDT,LUNAUSDT,SOLUSDT,XRPUSDT,DOGEUSDT,APEUSDT,SHIBUSDT,...,VTHOUSDT,ONGUSDT,OXTUSDT,AMPUSDT,AVAUSDT,WANUSDT,ALCXUSDT,DFUSDT,OMUSDT,XNOUSDT
datetime,,,,,,,,,,,,,,,,,,,,,
2025-12-31 23:55:00,11.381136,7.996930,-4.190415,6.762036,-2.350725,4.825670,0.611503,-2.141572,-1.627093,-11.881095,...,-7.192774,-2.475749,-3.79424,-6.285030,-1.276902,-2.632479,2.008214,-4.478538,-2.672201,-0.372514
2025-12-31 23:56:00,11.381167,7.997266,-4.190415,6.762313,-2.349677,4.825991,0.611612,-2.141232,-1.627093,-11.881095,...,-7.192774,-2.475749,-3.79424,-6.276484,-1.276902,-2.635265,2.008214,-4.480301,-2.672201,-0.372514
2025-12-31 23:57:00,11.381026,7.996869,-4.189755,6.762163,-2.351775,4.825429,0.610961,-2.141998,-1.627602,-11.882541,...,-7.192774,-2.476938,-3.79424,-6.277016,-1.276902,-2.632479,2.010895,-4.482068,-2.672201,-0.372514
2025-12-31 23:58:00,11.381026,7.996872,-4.191076,6.762105,-2.351775,4.825349,0.611069,-2.141742,-1.627602,-11.882541,...,-7.192774,-2.475749,-3.79424,-6.273297,-1.276902,-2.632479,2.010895,-4.483838,-2.672201,-0.372514
2025-12-31 23:59:00,11.381086,7.996869,-4.191076,6.761920,-2.350725,4.825510,0.611015,-2.141572,-1.627602,-11.882541,...,-7.192774,-2.476938,-3.79424,-6.273297,-1.279056,-2.632479,2.010895,-4.482953,-2.670754,-0.373966


Saved: crypto_universe_output\log_price_2023_2025_top200.parquet


In [8]:
# 불러올 때.
log_price_panel = pd.read_parquet(
    f"{FILTER_OUTPUT_DIR}/log_price_2023_2025_top200.parquet"
)

log_price_panel.index = pd.to_datetime(log_price_panel.index)

print(log_price_panel.shape)
display(log_price_panel.head())

(1576800, 200)


,BTCUSDT,ETHUSDT,GMTUSDT,BNBUSDT,LUNAUSDT,SOLUSDT,XRPUSDT,DOGEUSDT,APEUSDT,SHIBUSDT,...,VTHOUSDT,ONGUSDT,OXTUSDT,AMPUSDT,AVAUSDT,WANUSDT,ALCXUSDT,DFUSDT,OMUSDT,XNOUSDT
datetime,,,,,,,,,,,,,,,,,,,,,
2023-01-01 00:00:00,9.713758,7.086847,-1.468372,5.506550,0.231905,2.300583,-1.081755,-2.654699,1.291159,-11.724882,...,-7.008681,-1.546933,-2.695628,-5.792614,-0.638659,-1.714798,2.617396,-3.26492,-3.568433,-0.450986
2023-01-01 00:01:00,9.713495,7.086813,-1.468372,5.506550,0.233411,2.303585,-1.081755,-2.654557,1.290609,-11.724882,...,-7.007576,-1.546933,-2.695628,-5.792614,-0.638659,-1.714243,2.617396,-3.26492,-3.568433,-0.450986
2023-01-01 00:02:00,9.713321,7.086613,-1.470111,5.506144,0.231826,2.303585,-1.082641,-2.655837,1.290609,-11.724882,...,-7.007576,-1.546933,-2.695628,-5.792614,-0.638659,-1.712024,2.617396,-3.26492,-3.568433,-0.450986
2023-01-01 00:03:00,9.713153,7.086587,-1.469676,5.505332,0.231826,2.303585,-1.082641,-2.656692,1.290334,-11.724882,...,-7.007576,-1.550226,-2.695628,-5.792614,-0.638659,-1.712024,2.617396,-3.26492,-3.568433,-0.450986
2023-01-01 00:04:00,9.713258,7.087006,-1.467071,5.506144,0.231270,2.302585,-1.082936,-2.656407,1.289783,-11.723646,...,-7.007576,-1.550226,-2.697110,-5.792614,-0.638659,-1.712024,2.617396,-3.26492,-3.568433,-0.454130
